# 03 - Embeddings + Chroma

**Goal:** turn each chunk into a list of numbers (an *embedding*) so we can search by meaning, not just by matching words.

An **embedding** is a point in space. A model reads a text and outputs a list of 384 numbers. Texts with similar meaning land close together, even in different languages. We measure "close" with **cosine similarity**: 1.0 = same direction (same meaning), near 0 = unrelated.

In [1]:
import os
from pathlib import Path

# The notebook may be started from the project root or from notebooks/.
if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)

import numpy as np

from assistant.config import get_settings
from assistant.index import chunk_to_documents, load_index, make_embeddings
from assistant.parse import read_chunks

settings = get_settings()
# multilingual-e5-small on CPU; adds the "passage: " / "query: " prefixes for us
emb = make_embeddings(settings)
chunks = read_chunks(settings.chunks_path)
by_id = {c.id: c for c in chunks}
len(chunks)

/Users/poojashreniaddisherla/Desktop/Pooja Job Stuff/Projects/eu-ai-gdpr-assistant/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


W1123 19:04:19.173000 10933 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 13156.99it/s]

3044

## 1. Same paragraph in German and English, plus an unrelated one

GDPR Art. 6(1) in both languages, and for contrast a paragraph that has nothing to do with lawful processing (Art. 94, repeal of an old directive). The vectors are normalized (length 1), so cosine similarity is just a dot product.

In [2]:
en = by_id["GDPR-en-article-6-1"].text
de = by_id["GDPR-de-article-6-1"].text
other = next(c.text for c in chunks if c.id == "GDPR-en-article-94-1")
vecs = np.array(emb.embed_documents([en, de, other]))
print("vector length:", vecs.shape[1], "| norms:", np.linalg.norm(vecs, axis=1).round(3))
print("EN vs DE (same article):  ", round(float(vecs[0] @ vecs[1]), 3))
print("EN vs unrelated (Art. 94):", round(float(vecs[0] @ vecs[2]), 3))
print("DE vs unrelated (Art. 94):", round(float(vecs[1] @ vecs[2]), 3))

vector length: 384 | norms: [1. 1. 1.]
EN vs DE (same article):   0.869
EN vs unrelated (Art. 94): 0.809
DE vs unrelated (Art. 94): 0.769


The German and English texts of Art. 6(1) are closer to each other (0.87) than to the unrelated paragraph (0.81 / 0.77). e5 models give high scores overall (even unrelated texts score ~0.8), so look at the *gap*, not the absolute number.

## 2. A German question: what comes back?

We search the Chroma index built by `python -m assistant.index`. The question is embedded with the `query: ` prefix; stored chunks were embedded with `passage: `.

In [3]:
index = load_index(emb, settings.chroma_dir)
q = "Wann ist die Verarbeitung personenbezogener Daten rechtmäßig?"
for doc, dist in index.similarity_search_with_score(q, k=5):
    m = doc.metadata
    label = f"{m['regulation']} {m['type']} {m['number']}({m['paragraph']})"
    print(f"{m['language']}  {label}  cosine distance={dist:.3f}")

print("--- same question, English chunks only ---")
for doc, dist in index.similarity_search_with_score(q, k=5, filter={"language": "en"}):
    m = doc.metadata
    label = f"{m['regulation']} {m['type']} {m['number']}({m['paragraph']})"
    print(f"{m['language']}  {label}  cosine distance={dist:.3f}")

de  GDPR recital 69()  cosine distance=0.108
de  GDPR article 6(1)  cosine distance=0.113
de  GDPR article 6(1)  cosine distance=0.116
de  GDPR article 10()  cosine distance=0.119
de  GDPR recital 46()  cosine distance=0.121
--- same question, English chunks only ---


en  GDPR recital 63()  cosine distance=0.176
en  AI_ACT recital 10()  cosine distance=0.177
en  GDPR article 21(2)  cosine distance=0.179
en  GDPR recital 63()  cosine distance=0.180
en  AI_ACT recital 39()  cosine distance=0.181


Two things to notice:

* Without a filter, a German question returns only **German** chunks: same-language text scores clearly higher (distance about 0.11 vs 0.18 for English). Embeddings are *shared* across languages, but not perfectly language-blind.
* With the `language="en"` filter we do get English chunks back, so cross-language search works, but these matches are looser (they are loosely about data subject rights and processing, and the English twin of Art. 6(1) is not in the top 5).

So for a German question we mainly rely on the German chunks, and the English ones act as a backup. In stage 4 we will mix in keyword search (BM25) and de-duplicate DE/EN twins so the model never reads the same provision twice.

## 3. Long chunks are split

e5 reads at most 512 tokens (roughly 1500-2000 characters); anything longer would be silently cut off. So chunks over 1200 characters are split, and **every part repeats the heading** so it makes sense on its own.

In [4]:
longest = max(chunks, key=lambda c: len(c.text))
parts = chunk_to_documents(longest)
print(longest.id, "-", len(longest.text), "chars ->", len(parts), "parts")
for p in parts[:3]:
    print(p.id, len(p.page_content), "|", p.page_content[:70].replace("\n", " / "), "...")

AI_ACT-de-article-5-1 - 6453 chars -> 7 parts
AI_ACT-de-article-5-1#p0 820 | Artikel 5 – Verbotene Praktiken im KI-Bereich / (1) Folgende Praktiken i ...
AI_ACT-de-article-5-1#p1 1158 | Artikel 5 – Verbotene Praktiken im KI-Bereich / b) das Inverkehrbringen, ...
AI_ACT-de-article-5-1#p2 1239 | Artikel 5 – Verbotene Praktiken im KI-Bereich / bb) das Inverkehrbringen ...


In [5]:
n_docs = len(index.get()["ids"])
extra = n_docs - len(chunks)
print(f"{len(chunks)} chunks -> {n_docs} documents in Chroma ({extra} extra from splitting)")

3044 chunks -> 3638 documents in Chroma (594 extra from splitting)
